# Fine tune SBERT 

In [ ]:
import requests
import pickle
from io import BytesIO
import logging

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

/opt/conda/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
record_id = "15583249"
access_token = "AUn57Ykztzzg7Cy8MdtAFEzGc53A4StqkVAYehmIa8lXe03evN1rHvV7GwZ6"  # Put your token here

api_url = f"https://zenodo.org/api/records/{record_id}"
headers = {"Authorization": f"Bearer {access_token}"}

# Get metadata
response = requests.get(api_url, headers=headers)
response.raise_for_status()
metadata = response.json()

# Find .pkl file and get the 'self' link
file_url = None
filename = None
for file in metadata.get('files', []):
    if file['key'].endswith('.pkl'):
        filename = file['key']
        file_url = file['links']['self']
        break

if not file_url:
    raise FileNotFoundError("No .pkl file found in the record.")

print(f"Downloading {filename} from {file_url}")

# Download file with auth header
file_response = requests.get(file_url, headers=headers)
file_response.raise_for_status()

# Load pickle from bytes
data = pickle.load(BytesIO(file_response.content))

print("Pickle file loaded:", type(data))


Pickle file loaded: <class 'list'>


In [3]:
# split a set out for evaluation
train_pairs, val_pairs = train_test_split(data, test_size=0.1, random_state=42)

In [4]:
# summary stats of train_examples
print(f"Number of training examples: {len(train_pairs)}")

Number of training examples: 180000


In [5]:
# Configure logging
logging.basicConfig(format='%(asctime)s - %(message)s',
                    level=logging.INFO)

In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

2025-06-03 11:25:14,944 - Use pytorch device_name: cuda:0
2025-06-03 11:25:14,948 - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


In [7]:
evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
    val_pairs, name='validation'
)

In [8]:
model.evaluate(evaluator)

2025-06-03 11:25:18,386 - EmbeddingSimilarityEvaluator: Evaluating the model on the validation dataset:
2025-06-03 11:25:37,443 - Cosine-Similarity :	Pearson: 0.3690	Spearman: 0.3635


{'validation_pearson_cosine': 0.3689586166396692,
 'validation_spearman_cosine': 0.3634638764472133}

In [9]:
# Check if GPU is available and move model to GPU
if torch.cuda.is_available():
    model = model.to('cuda')
    print("Using GPU for training")
else:
    print("GPU not available, using CPU")

Using GPU for training


In [10]:
train_dataloader = DataLoader(
    train_pairs,
    shuffle=True,
    batch_size=32,
    num_workers=4  # Adjust based on your CPU
)

/opt/conda/envs/nlp_env_analysis/lib/python3.10/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [11]:
train_dataloader = DataLoader(train_pairs, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model=model)

In [12]:
model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=2)

Step,Training Loss
500,0.109800
1000,0.091900
1500,0.075100
2000,0.063100
2500,0.052700
3000,0.042900
3500,0.037000
4000,0.032300
4500,0.025900
5000,0.021400


In [14]:
model.push_to_hub(
    "objection_fine_tuned"
    )

2025-06-03 12:54:44,523 - Save model to /tmp/tmpo3z_rkcx


'https://huggingface.co/Bea-Taylor/objection_fine_tuned/commit/1da2e120163f81be248e9f55e702ee2dde6aa8d7'

In [17]:
fine_tuned_model = SentenceTransformer("Bea-Taylor/objection_fine_tuned")
fine_tuned_model.evaluate(evaluator)

2025-06-03 12:57:32,515 - Use pytorch device_name: cuda:0
2025-06-03 12:57:32,517 - Load pretrained SentenceTransformer: Bea-Taylor/objection_fine_tuned
2025-06-03 12:57:39,128 - EmbeddingSimilarityEvaluator: Evaluating the model on the validation dataset:
2025-06-03 12:57:58,020 - Cosine-Similarity :	Pearson: 0.9903	Spearman: 0.9212


{'validation_pearson_cosine': 0.9902688920867188,
 'validation_spearman_cosine': 0.9211631394910035}